# Drift Ainwater - Ejecución de Reportes
**Autores:** Franco Chiappe, Vicente Garay, Ziyu Guo  


In [4]:
from pathlib import Path
import pandas as pd
from Funciones_Drift import make_report_for_plant, run_drift_batch

plant_files = {
    'planta1': Path('../df_procesados/df_planta_1.csv'),
    'planta2': Path('../df_procesados/df_planta_2.csv'),
    'planta3': Path('../df_procesados/df_planta_3.csv'),
}
flag_files = {
    'planta1': Path('../df_procesados/flags_p1.csv'),
    'planta2': Path('../df_procesados/flags_p2.csv'),
    'planta3': Path('../df_procesados/flags_p3.csv'),
}

output_dir_base = Path('../ReportesDrift')
output_dir_base.mkdir(parents=True, exist_ok=True)
output_dirs = {plant: output_dir_base/plant for plant in plant_files}
for d in output_dirs.values(): d.mkdir(parents=True, exist_ok=True)

#
RUN_PLANTS = ["planta1", "planta2", "planta3"]

def iter_plants():
    if RUN_PLANTS:
        for p in RUN_PLANTS:
            if p in plant_files:
                yield p, plant_files[p]
    else:
        for p, path in plant_files.items():
            yield p, path

## Parámetros generales

In [5]:
# --- Parámetros de ventanas ---
CURRENT_WINDOW: str = '3D' # Ventana Actual('3D', '72H')

# --- Parámetros de resampleo ---
RESAMPLE: str | None = None # 5min, 10min | Se puede hacer resample o trabajar con todos los Datos
RESAMPLE_AGG: str = 'mean' # Como se calcula el sample('mean', 'median')

# --- Columnas a excluir ---
EXCLUDE_COLUMNS: list[str] = ['pH Ecualizador 2 (Tk 250m3)', "Conductividad DAF",
                               "Temperatura DAF", 'pH entrada a Ecualizador 1', #Planta1
 "Flujo Aire Reactor 1", "OD Reactor 1"] # Columnas que dan problemas o no aportan | Planta2

# --- Calidad mínima de datos por columna/ventana ---
"""MIN_N: int = 30  # mínimo de observaciones no nulas
COV_MIN: float = 0.05  # cobertura mínima
"""
# --- Método estadístico para drift ---
NUM_METHOD: str = 'auto' # 'auto', 'ks', 'wasserstein', etc...
NUM_THRESHOLD: float | None = None # Umbral para definir Drift

# --- Parámetros Decay ---
DECAY_HALF_LIFE_HOURS: int = 24*7 # mayor ---> más robusto pero reacciona más lento
DECAY_WEIGHT_MASS: float = 0.95 # toma lo más reciente hasta cubrir en 95% del peso

# --- Parámetros Golden ---
GOLDEN_WIN: str = '30min'
GOLDEN_STEP: str = '10min'
GOLDEN_K: int = 40 # mayor ---> más robusto pero reacciona más lento

# --- Parámetros Seasonal ---
SEASONAL_WEEKS_BACK: int = 12

In [6]:
plants = ["planta1", "planta2", "planta3"]
strategies = ["decay", "golden", "seasonal"]
stadistics = [ 'ks', 'mannwhitney', 'psi', 'wasserstein']


paths, errs = run_drift_batch(
    plant_names=plants,
    strategies=strategies,
    plant_files=plant_files,
    flag_files=flag_files,
    output_root=Path("../ReportesDrift"),
    CURRENT_WINDOW="3D",
    RESAMPLE=RESAMPLE,
    RESAMPLE_AGG=RESAMPLE_AGG,
    EXCLUDE_COLUMNS=EXCLUDE_COLUMNS,
    NUM_METHOD=NUM_METHOD,
    NUM_THRESHOLD=NUM_THRESHOLD,
)

[OK] planta1 · decay → planta1_decay.html
[OK] planta1 · golden → planta1_golden.html
[OK] planta1 · seasonal → planta1_seasonal.html
[OK] planta2 · decay → planta2_decay.html
[OK] planta2 · golden → planta2_golden.html
[OK] planta2 · seasonal → planta2_seasonal.html
[OK] planta3 · decay → planta3_decay.html
[OK] planta3 · golden → planta3_golden.html
[OK] planta3 · seasonal → planta3_seasonal.html
